In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

df = pd.read_parquet('../data/processed/protein_features_hol.parquet')

VALID_START, TEST_START = "2017-07-15", "2017-07-31"

CATEGORICALS = ["family", "city", "state", "store_type", "store_nbr", "item_nbr", "class", "cluster"]
for c in CATEGORICALS:
    df[c] = df[c].astype("category")

valid = df[(df["date"] >= VALID_START) & (df["date"] < TEST_START)].copy()

model = lgb.Booster(model_file='../models/lgbm_holidays.txt')
FEATURES = [c for c in df.columns if c not in ["date", "unit_sales"]]

valid["pred_lgbm"] = np.clip(np.expm1(model.predict(valid[FEATURES])), 0, None)
valid["pred_baseline"] = valid["roll_mean_7"]

print("Valid rows:", len(valid))
print(valid[["date", "family", "unit_sales", "pred_lgbm", "pred_baseline"]].head())

Valid rows: 129573
          date family  unit_sales  pred_lgbm  pred_baseline
896 2017-07-15   DELI         1.0   1.071110       1.142857
897 2017-07-16   DELI         0.0   0.550397       1.142857
898 2017-07-17   DELI         1.0   1.454521       1.142857
899 2017-07-18   DELI         3.0   1.064335       1.142857
900 2017-07-19   DELI         1.0   1.174696       1.571429


In [2]:
# Assumptions — documented, not hidden
UNIT_PRICE   = 8.00   # avg retail price per unit (kg) of protein
MARGIN_PCT   = 0.25   # gross margin
COGS         = UNIT_PRICE * (1 - MARGIN_PCT)   # $6.00 cost of goods
MARGIN       = UNIT_PRICE * MARGIN_PCT         # $2.00 profit per unit

# Over-forecast: excess spoils, lose full cost
COST_OVER  = COGS      # $6.00 per excess unit
# Under-forecast: lost sale, forgo margin
COST_UNDER = MARGIN    # $2.00 per unit short

print(f"Cost per unit over-forecast:  ${COST_OVER:.2f}  (spoilage)")
print(f"Cost per unit under-forecast: ${COST_UNDER:.2f}  (lost margin)")
print(f"Asymmetry ratio: {COST_OVER/COST_UNDER:.1f}x")

Cost per unit over-forecast:  $6.00  (spoilage)
Cost per unit under-forecast: $2.00  (lost margin)
Asymmetry ratio: 3.0x


In [3]:
# the cost function:

def forecast_cost(actual, forecast, cost_over=COST_OVER, cost_under=COST_UNDER):
    """Asymmetric cost: over-forecasting spoils inventory, under-forecasting loses sales."""
    error = forecast - actual
    over  = np.clip(error, 0, None)      # forecast exceeded actual
    under = np.clip(-error, 0, None)     # actual exceeded forecast
    return over * cost_over + under * cost_under

valid["cost_lgbm"]     = forecast_cost(valid["unit_sales"], valid["pred_lgbm"])
valid["cost_baseline"] = forecast_cost(valid["unit_sales"], valid["pred_baseline"])

total_lgbm = valid["cost_lgbm"].sum()
total_base = valid["cost_baseline"].sum()
saving = total_base - total_lgbm

print(f"16-day validation period, 54 stores, 263 items\n")
print(f"Baseline (moving avg) cost: ${total_base:,.0f}")
print(f"LightGBM cost:              ${total_lgbm:,.0f}")
print(f"Saving:                     ${saving:,.0f}  ({saving/total_base*100:.1f}%)")
print(f"\nAnnualized: ${saving * 365/16:,.0f}")

16-day validation period, 54 stores, 263 items

Baseline (moving avg) cost: $2,022,885
LightGBM cost:              $1,370,613
Saving:                     $652,272  (32.2%)

Annualized: $14,879,950


In [4]:
# break down where the cost comes from (decomposition)

for name, pred_col in [("Baseline", "pred_baseline"), ("LightGBM", "pred_lgbm")]:
    err = valid[pred_col] - valid["unit_sales"]
    over_units  = np.clip(err, 0, None).sum()
    under_units = np.clip(-err, 0, None).sum()
    print(f"\n{name}:")
    print(f"  Over-forecast units:  {over_units:>10,.0f}  →  ${over_units*COST_OVER:>10,.0f} spoilage")
    print(f"  Under-forecast units: {under_units:>10,.0f}  →  ${under_units*COST_UNDER:>10,.0f} lost margin")
    print(f"  Spoilage share of total cost: {over_units*COST_OVER/(over_units*COST_OVER+under_units*COST_UNDER)*100:.1f}%")


Baseline:
  Over-forecast units:     248,356  →  $ 1,490,138 spoilage
  Under-forecast units:    266,374  →  $   532,747 lost margin
  Spoilage share of total cost: 73.7%

LightGBM:
  Over-forecast units:     131,918  →  $   791,511 spoilage
  Under-forecast units:    289,551  →  $   579,102 lost margin
  Spoilage share of total cost: 57.7%


In [5]:
# Sensitivity test, so the conclusion doesn't rest on one guess:

print("Cost reduction % under different assumptions:\n")
print(f"{'Price':>7} {'Margin':>8} {'Ratio':>7} {'Reduction':>11}")
print("-" * 36)

for price in [5.0, 8.0, 12.0]:
    for margin_pct in [0.15, 0.25, 0.40]:
        c_over  = price * (1 - margin_pct)
        c_under = price * margin_pct
        b = forecast_cost(valid["unit_sales"], valid["pred_baseline"], c_over, c_under).sum()
        l = forecast_cost(valid["unit_sales"], valid["pred_lgbm"], c_over, c_under).sum()
        print(f"${price:>6.2f} {margin_pct:>7.0%} {c_over/c_under:>6.1f}x {(b-l)/b*100:>10.1f}%")

Cost reduction % under different assumptions:

  Price   Margin   Ratio   Reduction
------------------------------------
$  5.00     15%    5.7x       38.0%
$  5.00     25%    3.0x       32.2%
$  5.00     40%    1.5x       23.7%
$  8.00     15%    5.7x       38.0%
$  8.00     25%    3.0x       32.2%
$  8.00     40%    1.5x       23.7%
$ 12.00     15%    5.7x       38.0%
$ 12.00     25%    3.0x       32.2%
$ 12.00     40%    1.5x       23.7%


In [6]:
# bias optimization 
# Spoilage is still 57.7% of your remaining cost, which says the model is systematically over-ordering relative to what's economically optimal. 
# Since over-errors cost 3×, deliberately shading forecasts downward should reduce total cost even though it makes accuracy metrics worse:

print("Effect of multiplying forecasts by a bias factor:\n")
print(f"{'Factor':>7} {'Cost':>12} {'vs 1.0':>9} {'WAPE':>7}")
print("-" * 38)

best = (None, np.inf)
for f in [0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00, 1.05]:
    adj = valid["pred_lgbm"] * f
    cost = forecast_cost(valid["unit_sales"], adj).sum()
    w = np.sum(np.abs(valid["unit_sales"] - adj)) / np.sum(valid["unit_sales"]) * 100
    if cost < best[1]:
        best = (f, cost)
    marker = ""
    print(f"{f:>7.2f} ${cost:>11,.0f} {(cost/1370613-1)*100:>8.1f}% {w:>6.1f}")

print(f"\nOptimal factor: {best[0]:.2f} → ${best[1]:,.0f}")
print(f"Additional saving vs unbiased: ${1370613 - best[1]:,.0f}")

Effect of multiplying forecasts by a bias factor:

 Factor         Cost    vs 1.0    WAPE
--------------------------------------
   0.70 $  1,249,361     -8.8%   50.7
   0.75 $  1,237,564     -9.7%   48.3
   0.80 $  1,236,945     -9.8%   46.1
   0.85 $  1,248,667     -8.9%   44.3
   0.90 $  1,274,290     -7.0%   42.8
   0.95 $  1,314,840     -4.1%   41.7
   1.00 $  1,370,613     -0.0%   40.9
   1.05 $  1,442,055      5.2%   40.5

Optimal factor: 0.80 → $1,236,945
Additional saving vs unbiased: $133,668


In [7]:
# breakdown by family, then save:

valid["pred_optimal"] = valid["pred_lgbm"] * 0.80
valid["cost_optimal"] = forecast_cost(valid["unit_sales"], valid["pred_optimal"])

summary = valid.groupby("family", observed=True).agg(
    units_sold=("unit_sales", "sum"),
    cost_baseline=("cost_baseline", "sum"),
    cost_lgbm=("cost_lgbm", "sum"),
    cost_optimal=("cost_optimal", "sum"),
).round(0)
summary["reduction_pct"] = ((summary["cost_baseline"] - summary["cost_optimal"]) / summary["cost_baseline"] * 100).round(1)

print(summary.to_string())

valid.to_parquet('../data/processed/valid_with_costs.parquet', index=False)

                units_sold  cost_baseline  cost_lgbm  cost_optimal  reduction_pct
family                                                                           
DELI              270728.0       564151.0   410501.0      362302.0           35.8
MEATS             325969.0       620178.0   380564.0      348197.0           43.9
POULTRY           339124.0       685356.0   464373.0      422555.0           38.3
PREPARED FOODS     77276.0       119353.0    90106.0       82236.0           31.1
SEAFOOD            17300.0        33847.0    25069.0       21656.0           36.0
